# Sistema RAG para normativa de la Universidad de El Salvador

**Notebook de demostración para la defensa técnica.** Presenta el pipeline final de forma breve y trazable; no sustituye a `LaboratorioML.ipynb`, que conserva el desarrollo experimental completo.

> Las celdas se entregan sin outputs simulados. Para ejecutarlas se necesitan las dependencias de `requirements.txt` y acceso local a los modelos descargados desde Hugging Face.

## 1. Objetivo de la demostración

Mostrar, paso a paso, cómo el sistema carga normativa institucional, conserva la procedencia de cada fragmento, recupera evidencia mediante MPNet y FAISS, y entrega esa evidencia a BETO-SQAC para extraer una respuesta.

## 2. ¿Qué problema resuelve el sistema?

Un PDF completo puede superar el contexto útil del modelo QA. El sistema divide el corpus en fragmentos y usa búsqueda semántica para seleccionar únicamente los diez candidatos más relacionados con la pregunta. Así reduce el espacio sometido a QA y conserva documento, página y chunk como fuente.

En este laboratorio, RAG no termina en un LLM generativo: BETO-SQAC realiza **Question Answering extractivo** y selecciona un span que ya existe en el texto recuperado.

## 3. Arquitectura general

### Fase de indexación

```text
PDF → extracción → limpieza → chunking → fragmentos + metadata
    → MPNet → embeddings normalizados → FAISS IndexFlatIP
```

### Fase de consulta

```text
Pregunta → MPNet → embedding de la pregunta → FAISS
         → Top-K chunks → BETO-SQAC → candidatos
         → selección por score QA → respuesta + fuente
```

La similitud de recuperación decide qué fragmentos llegan a QA. El score QA decide qué span candidato se presenta como respuesta. Son valores distintos.

## 4. Configuración final

| Componente | Configuración validada |
|---|---|
| Modelo QA | `MMG/bert-base-spanish-wwm-cased-finetuned-sqac` |
| Embeddings | `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` |
| Chunk size | 800 caracteres |
| Overlap | 0 |
| Top-K | 10 |
| Índice | FAISS `IndexFlatIP` |

Estos valores provienen del laboratorio ejecutado; la demo no vuelve a optimizarlos.

In [ ]:
import json
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sentence_transformers import SentenceTransformer
from transformers import pipeline

from src.funciones_qa import (
    buscar_chunks_multidocumento,
    construir_indice_faiss_multidocumento,
    crear_chunks_multidocumento,
    extraer_documento_pdf,
    rag_multidocumento,
    validar_metadata_chunks,
)

MODELO_QA = "MMG/bert-base-spanish-wwm-cased-finetuned-sqac"
MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
CHUNK_SIZE = 800
OVERLAP = 0
TOP_K = 10

In [ ]:
configuracion = pd.DataFrame(
    {
        "Valor": [
            MODELO_QA,
            MODELO_EMBEDDINGS,
            CHUNK_SIZE,
            OVERLAP,
            TOP_K,
            "IndexFlatIP",
        ]
    },
    index=["QA", "Embeddings", "Chunk size", "Overlap", "Top-K", "FAISS"],
)
display(configuracion)

## 5. Carga del corpus

La primera demostración utiliza únicamente `data/documento_fuente.pdf`. Las rutas se resuelven desde la raíz del repositorio para mantener portabilidad.

In [ ]:
RAIZ = Path.cwd().resolve()
PDF_PRINCIPAL = RAIZ / "data" / "documento_fuente.pdf"
PREGUNTAS = RAIZ / "data" / "preguntas_multidocumento.json"

if not (RAIZ / "src" / "funciones_qa.py").is_file():
    raise RuntimeError("Abra el notebook desde la raíz del repositorio")
if not PDF_PRINCIPAL.is_file():
    raise FileNotFoundError(PDF_PRINCIPAL)

documento_mono = extraer_documento_pdf(PDF_PRINCIPAL, documento_id=1)
print(f"Documento cargado: {documento_mono['documento']}")

In [ ]:
resumen_documento = pd.DataFrame(
    [
        {
            "Documento": documento_mono["documento"],
            "Páginas": documento_mono["numero_paginas"],
            "Páginas con texto": documento_mono["paginas_con_texto"],
            "Caracteres": documento_mono["caracteres"],
            "Palabras aproximadas": documento_mono["palabras_aproximadas"],
        }
    ]
)
display(resumen_documento)

## 6. Extracción y preparación del texto

`extraer_documento_pdf` usa PyMuPDF, procesa cada página por separado y aplica una limpieza conservadora. Además del texto unido, conserva los offsets inicial y final de cada página. Esos offsets permiten asociar después un chunk con su página física.

In [ ]:
pagina_inicial = documento_mono["paginas"][0]
display(
    pd.DataFrame(
        [
            {
                "Página": pagina_inicial["pagina"],
                "Inicio": pagina_inicial["inicio"],
                "Fin": pagina_inicial["fin"],
                "Tiene texto": pagina_inicial["tiene_texto"],
                "Muestra": pagina_inicial["texto"][:240].replace("\n", " ") + "…",
            }
        ]
    )
)

## 7. Fragmentación del documento

Se crean ventanas consecutivas de 800 caracteres y overlap cero. La función reutilizada acepta una lista de documentos; una lista de un elemento representa el caso monodocumento.

In [ ]:
metadata_mono = crear_chunks_multidocumento(
    [documento_mono],
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
)

validacion_mono = validar_metadata_chunks(metadata_mono)
assert len(metadata_mono) == 217
print(f"Chunks creados: {len(metadata_mono)}")
print(f"Metadata alineada para {validacion_mono['chunks']} chunks")

In [ ]:
longitudes = np.array([chunk["num_caracteres"] for chunk in metadata_mono])
display(
    pd.DataFrame(
        [
            {
                "Chunks": len(metadata_mono),
                "Longitud mínima": int(longitudes.min()),
                "Longitud máxima": int(longitudes.max()),
                "Longitud promedio": float(longitudes.mean()),
            }
        ]
    ).style.format({"Longitud promedio": "{:.1f}"})
)

## 8. Metadata de los fragmentos

Cada vector tiene una fila de metadata con documento, página, chunk y posición vectorial. La igualdad `vector_posicion == posición en FAISS` es la base de la trazabilidad.

In [ ]:
tabla_metadata = pd.DataFrame(
    [
        {
            "Documento": chunk["documento"],
            "Página inicial": chunk["pagina_inicio"],
            "Página final": chunk["pagina_fin"],
            "Chunk": chunk["chunk_id"],
            "Posición FAISS": chunk["vector_posicion"],
            "Texto": chunk["texto"][:180].replace("\n", " ") + "…",
        }
        for chunk in metadata_mono[:5]
    ]
)
display(tabla_metadata)

## 9. Generación de embeddings

MPNet transforma cada texto en un vector de 768 dimensiones. Los vectores se normalizan a norma L2 igual a uno; por eso el producto interno usado por FAISS produce el mismo ranking que la similitud coseno.

En esta celda también se carga una sola vez el pipeline BETO-SQAC que se utilizará después.

In [ ]:
dispositivo_embeddings = "cuda" if torch.cuda.is_available() else "cpu"
dispositivo_qa = 0 if torch.cuda.is_available() else -1

modelo_embeddings = SentenceTransformer(
    MODELO_EMBEDDINGS,
    device=dispositivo_embeddings,
)
pipeline_qa = pipeline(
    "question-answering",
    model=MODELO_QA,
    tokenizer=MODELO_QA,
    device=dispositivo_qa,
)

print(f"Embeddings en: {dispositivo_embeddings}")
print(f"QA en: {'cuda' if dispositivo_qa == 0 else 'cpu'}")

In [ ]:
embeddings_mono, indice_mono, estadisticas_mono = (
    construir_indice_faiss_multidocumento(
        modelo_embeddings,
        metadata_mono,
        batch_size=32,
    )
)

normas = np.linalg.norm(embeddings_mono, axis=1)
assert embeddings_mono.shape == (217, 768)
assert np.allclose(normas, 1.0, atol=1e-5)

In [ ]:
display(
    pd.DataFrame(
        [
            {
                "Modelo": MODELO_EMBEDDINGS,
                "Matriz": str(embeddings_mono.shape),
                "Dimensión": embeddings_mono.shape[1],
                "Norma mínima": float(normas.min()),
                "Norma máxima": float(normas.max()),
            }
        ]
    ).style.format({"Norma mínima": "{:.6f}", "Norma máxima": "{:.6f}"})
)

## 10. Construcción del índice FAISS

`IndexFlatIP` almacena todos los vectores y realiza una búsqueda exacta por producto interno. No entrena centroides ni aproxima vecinos. La correspondencia entre posición vectorial y metadata se valida antes de consultar.

In [ ]:
assert indice_mono.ntotal == len(metadata_mono)
assert indice_mono.d == embeddings_mono.shape[1]
display(pd.DataFrame([estadisticas_mono]))

## 11. ¿Qué ocurre cuando llega una pregunta?

1. Se valida que la pregunta no esté vacía.
2. MPNet crea y normaliza su embedding.
3. FAISS devuelve los diez vectores más similares.
4. La metadata recupera documento, página, chunk y texto.
5. BETO-SQAC analiza cada uno de los diez textos.
6. Se selecciona el candidato con mayor **score QA**.
7. La respuesta se presenta con su fuente.

## 12. Embedding de la pregunta

La pregunta de ejemplo se toma del dataset existente; no se crea una etiqueta nueva.

In [ ]:
preguntas_existentes = json.loads(PREGUNTAS.read_text(encoding="utf-8"))
item_demo_mono = next(
    item
    for item in preguntas_existentes
    if item["documento_esperado"] == PDF_PRINCIPAL.name
    and not item["sin_respuesta"]
)
pregunta_demo = item_demo_mono["pregunta"]

print("Pregunta:")
print(pregunta_demo)

In [ ]:
embedding_pregunta = modelo_embeddings.encode(
    [pregunta_demo],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
).astype(np.float32)

display(
    pd.DataFrame(
        [
            {
                "Forma": str(embedding_pregunta.shape),
                "Dimensión": embedding_pregunta.shape[1],
                "Norma L2": float(np.linalg.norm(embedding_pregunta[0])),
            }
        ]
    ).style.format({"Norma L2": "{:.6f}"})
)

## 13. Recuperación semántica Top-K

La recuperación usa exclusivamente el embedding de la pregunta. En esta etapa todavía no se ha ejecutado BETO-SQAC.

In [ ]:
recuperados_mono = buscar_chunks_multidocumento(
    pregunta_demo,
    modelo_embeddings,
    indice_mono,
    metadata_mono,
    top_k=TOP_K,
)
assert len(recuperados_mono) == TOP_K

## 14. Visualización de fragmentos recuperados

La tabla separa el rank y la similitud de recuperación de cualquier score producido posteriormente por QA.

In [ ]:
def tabla_recuperacion(recuperados):
    """Convierte los candidatos recuperados en una tabla para la defensa."""
    return pd.DataFrame(
        [
            {
                "Rank": chunk["rank"],
                "Documento": chunk["documento"],
                "Página": chunk["pagina"],
                "Chunk": chunk["chunk_id"],
                "Similitud de recuperación": chunk["similitud"],
                "Muestra": chunk["texto"][:150].replace("\n", " ") + "…",
            }
            for chunk in recuperados
        ]
    )

display(
    tabla_recuperacion(recuperados_mono).style.format(
        {"Similitud de recuperación": "{:.6f}"}
    )
)

In [ ]:
rank_uno = recuperados_mono[0]
display(
    Markdown(
        "**Fragmento con mayor similitud de recuperación**  \n"
        f"Documento: `{rank_uno['documento']}` · "
        f"Página: {rank_uno['pagina']} · Chunk: {rank_uno['chunk_id']}\n\n"
        f"> {rank_uno['texto'][:700].replace(chr(10), ' ')}…"
    )
)

## 15. Question Answering sobre los fragmentos

BETO-SQAC recibe la misma pregunta junto con cada chunk Top-10 y extrae un span por candidato. La función de `src` conserva tanto la similitud de recuperación como el score QA.

In [ ]:
resultado_mono = rag_multidocumento(
    pregunta_demo,
    modelo_embeddings,
    indice_mono,
    metadata_mono,
    pipeline_qa,
    top_k=TOP_K,
)

In [ ]:
tabla_candidatos_qa = pd.DataFrame(
    [
        {
            "Rank recuperación": candidato["rank"],
            "Documento": candidato["documento"],
            "Página": candidato["pagina"],
            "Chunk": candidato["chunk_id"],
            "Similitud de recuperación": candidato["similitud"],
            "Respuesta candidata": candidato["answer"],
            "Score QA": candidato["score"],
        }
        for candidato in resultado_mono["candidatos_qa"]
    ]
)
display(
    tabla_candidatos_qa.style.format(
        {"Similitud de recuperación": "{:.6f}", "Score QA": "{:.6f}"}
    )
)

## 16. Selección de la mejor respuesta

El ganador es el candidato con mayor **score QA**, no necesariamente el chunk situado en el rank 1 de FAISS. La similitud controla recuperación; el score QA controla la selección del span.

In [ ]:
display(
    pd.DataFrame(
        [
            {
                "Rank del chunk ganador": resultado_mono["rank_fuente"],
                "Similitud de recuperación": resultado_mono[
                    "similitud_recuperacion"
                ],
                "Score QA": resultado_mono["score_qa"],
            }
        ]
    ).style.format(
        {"Similitud de recuperación": "{:.6f}", "Score QA": "{:.6f}"}
    )
)

## 17. Respuesta con trazabilidad

La salida final reúne la respuesta y los dos scores con documento, página y chunk. Esto permite revisar la evidencia original.

In [ ]:
def mostrar_respuesta(resultado):
    """Muestra una respuesta RAG sin confundir recuperación con QA."""
    display(
        Markdown(
            f"**Pregunta:** {resultado['pregunta']}\n\n"
            f"**Respuesta:** {resultado['respuesta']}\n\n"
            f"**Score QA:** {resultado['score_qa']:.6f}\n\n"
            f"**Similitud de recuperación:** "
            f"{resultado['similitud_recuperacion']:.6f}\n\n"
            f"**Fuente:** `{resultado['documento_fuente']}`\n\n"
            f"**Página:** {resultado['pagina_fuente']}  \n"
            f"**Chunk:** {resultado['chunk_fuente']}  \n"
            f"**Rank de recuperación:** {resultado['rank_fuente']}"
        )
    )

mostrar_respuesta(resultado_mono)

## 18. Extensión multidocumento

La extensión conserva cada PDF de forma independiente. Para que la defensa sea ágil, esta sección carga el índice y la metadata ya producidos por el experimento validado; no vuelve a vectorizar los 765 chunks ni modifica `results/`.

El flujo de consulta es el mismo, pero ahora la procedencia documental forma parte de la respuesta.

In [ ]:
rutas_pdf = [
    PDF_PRINCIPAL,
    *sorted((RAIZ / "data" / "corpus_complementario").glob("*.pdf")),
]
assert len(rutas_pdf) == 9
assert len({ruta.name for ruta in rutas_pdf}) == 9

ruta_metadata_multi = RAIZ / "results" / "rag_multidocumento_metadata.jsonl"
ruta_indice_multi = RAIZ / "results" / "indice_rag_multidocumento.faiss"

metadata_multi = [
    json.loads(linea)
    for linea in ruta_metadata_multi.read_text(encoding="utf-8").splitlines()
]
indice_multi = faiss.read_index(str(ruta_indice_multi))

validar_metadata_chunks(metadata_multi)
assert len(metadata_multi) == 765
assert indice_multi.ntotal == len(metadata_multi)
assert indice_multi.d == 768

In [ ]:
resumen_corpus = pd.read_csv(
    RAIZ / "results" / "rag_multidocumento_corpus.csv"
)
display(resumen_corpus[["documento", "paginas", "chunks"]])

print(f"PDF independientes: {len(rutas_pdf)}")
print(f"Vectores indexados: {indice_multi.ntotal}")
print(f"Dimensión: {indice_multi.d}")

In [ ]:
item_demo_multi = next(
    item
    for item in preguntas_existentes
    if item["documento_esperado"] != PDF_PRINCIPAL.name
    and not item["sin_respuesta"]
)

resultado_multi = rag_multidocumento(
    item_demo_multi["pregunta"],
    modelo_embeddings,
    indice_multi,
    metadata_multi,
    pipeline_qa,
    top_k=TOP_K,
)

display(
    tabla_recuperacion(resultado_multi["recuperados"]).style.format(
        {"Similitud de recuperación": "{:.6f}"}
    )
)
mostrar_respuesta(resultado_multi)

## 19. Demostración interactiva

La celda siguiente reutiliza el índice multidocumento. Puede escribirse una pregunta nueva durante la defensa o presionar Enter para usar la pregunta existente de ejemplo. La consulta no recalcula métricas ni escribe archivos.

In [ ]:
pregunta_usuario = input("Pregunta sobre normativa UES: ").strip()
if not pregunta_usuario:
    pregunta_usuario = item_demo_multi["pregunta"]

resultado_interactivo = rag_multidocumento(
    pregunta_usuario,
    modelo_embeddings,
    indice_multi,
    metadata_multi,
    pipeline_qa,
    top_k=TOP_K,
)

display(
    tabla_recuperacion(resultado_interactivo["recuperados"]).style.format(
        {"Similitud de recuperación": "{:.6f}"}
    )
)
mostrar_respuesta(resultado_interactivo)

## 20. Resumen del flujo

```text
CARGAR → FRAGMENTAR → VECTORIZAR → INDEXAR
                                  ↓
RESPUESTA + FUENTE ← SELECCIONAR ← QA ← TOP-K ← PREGUNTA
```

- **MPNet** representa preguntas y chunks en el mismo espacio vectorial.
- **FAISS** recupera los chunks semánticamente próximos.
- **BETO-SQAC** extrae una respuesta de cada candidato.
- La selección final usa **score QA** y conserva **similitud de recuperación**.
- Documento, página y chunk permiten volver a la evidencia.

Responsabilidades de los notebooks:

- `LaboratorioML.ipynb`: desarrollo, experimentos y evidencia canónica.
- `RAG_UES_Demo.ipynb`: explicación y demostración del pipeline final.
- `laboratorio_ml_qa.ipynb`: antecedente histórico.